In [1]:
import number_parser
import re

from collections import deque
from itertools import combinations

In [2]:
day11_inputs = "inputs/day11.txt"
with open(day11_inputs, 'r') as file:
    day11_data = file.readlines()
day11_data = [line.strip() for line in day11_data]

In [ ]:
initial_floors = {
    "F4": [],
    "F3": [],
    "F2": [],
    "F1": []
}

for floor_area_arrangement in day11_data:
    faa_split = floor_area_arrangement.split("contains")[1].split(",")
    faa_split = [part for item in faa_split for part in item.split(" and ")]
    curr_floor = number_parser.parse(floor_area_arrangement.split(' ')[1])
    
    for item in faa_split:
        if "microchip" in item:
            microchip_type = re.search(r"a (\w+)-compatible microchip", item).group(1)
            initial_floors[f"F{curr_floor}"].append(f"{microchip_type[:2].capitalize()}M")
        if "generator" in item:
            generator_type = re.search(r"a (\w+) generator", item).group(1)
            initial_floors[f"F{curr_floor}"].append(f"{generator_type[:2].capitalize()}G")

print(initial_floors)

{'F4': [], 'F3': [], 'F2': ['PoM', 'PrM'], 'F1': ['PoG', 'ThG', 'ThM', 'PrG', 'RuG', 'RuM', 'CoG', 'CoM']}


In [4]:
# defining the allowed rules
# - micrchips do not affect each other and can be brought up together in the elevator
# - microchips should not be left with a non-compatible generator
# - the elevator can carry, at most, two generators or microchips, in any combination
# - the eleavtor only functions if it contains at least one microchip or generator
# - the eleavtor always stops at each floor, so items in the elevator and on the floor can irradiate each other

# the goal is to have all the microchips and generators on the fourth floor

# I should approach this using a breadth-first search algorithm (BFS)
# each state is the elevator location and the items on each floor

In [5]:
def is_valid_state(floors):
    for floor in floors:
        generators = {item[:-1] for item in floor if item.endswith('G')}
        if generators:
            for item in floor:
                if item.endswith('M') and item[:-1] not in generators:
                    return False
    return True

def canonical(state):
    elevator, floors = state
    pairs = {}
    for i, floor in enumerate(floors):
        for item in floor:
            element = item[:-1]
            item_type = item[-1]
            if element not in pairs:
                pairs[element] = [None, None]
            pairs[element][0 if item_type == 'G' else 1] = i
    return (elevator, tuple(sorted(tuple(p) for p in pairs.values())))

In [ ]:
def generate_next_states(state):
    elevator, floors = state
    next_states = []
    
    current_floor_items = floors[elevator]
    possible_moves = list(combinations(current_floor_items, 1)) + list(combinations(current_floor_items, 2))
    
    for move in possible_moves:
        move_set = set(move)
        for direction in [-1, 1]:
            new_elevator = elevator + direction
            if not (0 <= new_elevator <= 3):
                continue
            
            new_floors = []
            for i, floor in enumerate(floors):
                if i == elevator:
                    new_floors.append(floor - move_set)
                elif i == new_elevator:
                    new_floors.append(floor | move_set)
                else:
                    new_floors.append(floor)
            new_floors = tuple(new_floors)
            
            if is_valid_state(new_floors):
                next_states.append((new_elevator, new_floors))
    return next_states

In [7]:
def bfs(initial_state):
    # track states and steps taken
    queue = deque([(initial_state, 0)])
    
    visited = set()
    visited.add(canonical(initial_state)) 
    
    total_items = sum(len(floor) for floor in initial_state[1])
    
    while queue:
        state, steps = queue.popleft()
        
        # check if all items are on the top floor
        if len(state[1][-1]) == total_items:
            return steps
        
        for next_state in generate_next_states(state):
            key = canonical(next_state)
            if key not in visited:
                visited.add(key)
                queue.append((next_state, steps + 1))
    return -1

# part1

In [8]:
floors = [frozenset(initial_floors[f"F{i}"]) for i in range(1, 5)]
initial_state = (0, tuple(floors))

result = bfs(initial_state)
print(f"minimum number of steps to bring all objects is {result}")

minimum number of steps to bring all objects is 47


# part2

In [9]:
floors = [frozenset(initial_floors[f"F{i}"]) for i in range(1, 5)]

# create a union of the sets and include the new elerium and dilithium generators/microchips
floors[0] = floors[0] | frozenset({"ElG", "ElM", "DiG", "DiM"})
initial_state = (0, tuple(floors))

result = bfs(initial_state)
print(f"minimum number of steps to bring all objects is {result}")

minimum number of steps to bring all objects is 71
